In [3]:
! pip install pyaudio SpeechRecognition


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [4]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, lfilter, freqz,stft 
import speech_recognition as sr
import pyaudio
from sklearn.preprocessing import StandardScaler
import nltk 
from nltk.sentiment import SentimentIntensityAnalyzer
nltk.download('vader_lexicon')


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/imac/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [6]:


recognizer = sr.Recognizer()

with sr.Microphone() as source:
    print("Say something...")
    audio = recognizer.listen(source)

try:
    text = recognizer.recognize_google(audio)
    print("You said: " + text)

    raw_audio = np.frombuffer(
        audio.get_raw_data(),
        dtype=np.int16
    )

    fs_audio = audio.sample_rate
    print("Sample rate:", fs_audio)

except sr.UnknownValueError:
    print("Could not understand the audio.")
except sr.RequestError as e:
    print("Speech recognition service error:", e)

Say something...
You said: Tera
Sample rate: 44100


In [7]:
sia = SentimentIntensityAnalyzer()
sentiment_scores = sia.polarity_scores(text)
print("Sentiment scores:", sentiment_scores)

Sentiment scores: {'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}


In [11]:
audio_signal=raw_audio/np.max(np.abs(raw_audio))
download_factor=max(1, int(fs_audio/8000))
downsampled_signal=audio_signal[::download_factor]

N=len(downsampled_signal)
fs_downsampled=fs_audio/download_factor
t=np.arange(N)/fs_downsampled
fc=fs_downsampled/8
rf_signal=downsampled_signal*np.cos(2*np.pi*fc*t)

In [12]:
I = rf_signal*np.cos(2*np.pi*fc*t)
Q= rf_signal*np.sin(2*np.pi*fc*t) 
def lpf(signal,kernel_size=101):
    kernel = np.ones(kernel_size)/kernel_size
    return np.convolve(signal, kernel, mode='same')
I_filtered = lpf(I)
Q_filtered = lpf(Q)
magnitude = np.sqrt(I_filtered**2 + Q_filtered**2)
phase = np.arctan2(Q_filtered, I_filtered)

In [13]:
f,t_stft,Zxx = stft(downsampled_signal, fs_downsampled, nperseg=256)
spectrogram = np.abs(Zxx)

In [17]:
n_frames = spectrogram.shape[1]
iq_time = np.linspace(0, 1, len(I_filtered))
frame_time = np.linspace(0, 1, n_frames)
iq_features = np.column_stack((
    np.interp(frame_time, iq_time, I_filtered),
    np.interp(frame_time, iq_time, Q_filtered)
))
spectrogram_features = spectrogram.T
combined_features=np.hstack((iq_features,spectrogram_features))
scaler = StandardScaler()
scaled_features = scaler.fit_transform(combined_features) 
print("Scaled features shape:", scaled_features.shape) 
print("Scaled features:", scaled_features)

Scaled features shape: (141, 131)
Scaled features: [[-0.05400427  0.06462785 -0.77550445 ... -1.0091505  -0.96315625
  -0.72429291]
 [ 0.01453678  0.02917971 -0.54185519 ... -0.88857825 -0.90726272
  -0.6815856 ]
 [-0.12465288  0.34625504 -0.42310067 ... -0.9523357  -0.93143355
  -0.68489491]
 ...
 [-0.39389281 -0.08309519  0.93267388 ... -0.47446635 -0.45418141
   0.14029217]
 [ 0.63709025  0.57535202  6.72473105 ...  0.65069696  0.04166743
   0.15811628]
 [ 1.27730808  0.20168168 -0.74995157 ... -0.87558547 -0.83729475
  -0.63561125]]


In [19]:
from sklearn.ensemble import RandomForestClassifier

In [20]:
X_dummy = np.random.rand(10, scaled_features.shape[1])
y_dummy = np.random.randint(0, 2, size=10)

clf=RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_dummy, y_dummy)
prediction=clf.predict(scaled_features)
print("Predictions:", prediction)

Predictions: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1
 0 0 0 1 1 0 0 0 1 1 1 0 0 0 1 1 1 1 1 0 0 0 0 0 1 0 0 1 1 1 1 1 1 1 0 0 0
 0 0 0 0 0 0 0 1 1 0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 1 0 0 0]
